[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/mp-1/blob/master/notebooks/02_nelder_mead.ipynb)

# 02. Метод Нелдера-Мида

Реализуем метод деформируемого симплекса и показываем последовательность симплексов.

In [ ]:
!pip install -q numpy pandas sympy matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

Q = np.array([[4.0, -2.0], [-2.0, 6.0]])
B = np.array([-8.0, 6.0])
C = 30.0
X0 = np.array([0.0, 0.0])
EPS = 1e-4
X_STAR = np.linalg.solve(Q, -B)
F_STAR = 21.6

def f(point):
    x = np.asarray(point, dtype=float)
    return float(0.5 * x @ Q @ x + B @ x + C)

def grad(point):
    x = np.asarray(point, dtype=float)
    return Q @ x + B

def original_formula(x, y):
    return 2 * (x - 3) ** 2 + 3 * (y + 2) ** 2 - 2 * x * y + 4 * x - 6 * y

def grid():
    x = np.linspace(-1.0, 4.0, 220)
    y = np.linspace(-3.0, 2.0, 220)
    xx, yy = np.meshgrid(x, y)
    return xx, yy, original_formula(xx, yy)

def plot_path(points, title, simplexes=None, out=None):
    xx, yy, zz = grid()
    fig, ax = plt.subplots(figsize=(7, 5), dpi=130)
    contour = ax.contour(xx, yy, zz, levels=25, cmap="viridis")
    ax.clabel(contour, inline=True, fontsize=7)
    if simplexes is not None:
        for simplex in simplexes:
            closed = np.vstack([simplex, simplex[0]])
            ax.plot(closed[:, 0], closed[:, 1], color="#4b5563", alpha=0.35, linewidth=1)
    points = np.asarray(points)
    ax.plot(points[:, 0], points[:, 1], marker="o", linewidth=1.8, markersize=3.5, label="траектория")
    ax.scatter([points[0, 0]], [points[0, 1]], color="#2563eb", s=45, label="x0")
    ax.scatter([X_STAR[0]], [X_STAR[1]], color="red", marker="*", s=120, label="x*")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    if out:
        fig.savefig(out, bbox_inches="tight")
    plt.show()

In [ ]:
def nelder_mead(func, x0, step=1.0, tol=1e-4, max_iter=300, alpha=1.0, gamma=2.0, beta=0.5, sigma=0.5):
    x0 = np.asarray(x0, dtype=float)
    simplex = [x0]
    for i in range(len(x0)):
        vertex = x0.copy()
        vertex[i] += step
        simplex.append(vertex)
    simplex = np.array(simplex, dtype=float)
    history = []

    for iteration in range(max_iter + 1):
        values = np.array([func(vertex) for vertex in simplex])
        order = np.argsort(values)
        simplex = simplex[order]
        values = values[order]
        spread = float(np.max(values) - np.min(values))
        history.append({
            "iter": iteration,
            "simplex": simplex.copy(),
            "best_x": simplex[0].copy(),
            "best_f": float(values[0]),
            "spread": spread,
        })
        if spread < tol:
            break

        centroid = simplex[:-1].mean(axis=0)
        worst = simplex[-1]
        reflected = centroid + alpha * (centroid - worst)
        f_reflected = func(reflected)

        if f_reflected < values[0]:
            expanded = centroid + gamma * (reflected - centroid)
            simplex[-1] = expanded if func(expanded) < f_reflected else reflected
        elif f_reflected < values[-2]:
            simplex[-1] = reflected
        else:
            if f_reflected < values[-1]:
                contracted = centroid + beta * (reflected - centroid)
            else:
                contracted = centroid - beta * (centroid - worst)
            if func(contracted) < min(f_reflected, values[-1]):
                simplex[-1] = contracted
            else:
                best = simplex[0].copy()
                simplex = best + sigma * (simplex - best)

    best = history[-1]["best_x"]
    return best, float(func(best)), history

In [ ]:
x_min, f_min, history = nelder_mead(f, X0, tol=EPS)
print("x =", x_min, "f =", f_min, "iterations =", history[-1]["iter"])

rows = []
for item in history[:8] + [history[-1]]:
    x = item["best_x"]
    rows.append([item["iter"], x[0], x[1], item["best_f"], item["spread"]])
df = pd.DataFrame(rows, columns=["k", "x", "y", "f_best", "spread"])
df.to_csv("tables/tab_4_1_nm.csv", index=False)
display(df)

points = np.array([item["best_x"] for item in history])
simplexes = [item["simplex"] for item in history[:: max(1, len(history) // 12)]]
plot_path(points, "Метод Нелдера-Мида", simplexes, "figures/fig_4_1_nm_trajectory.png")